In [ ]:
# Preparation
import pandas as pd, sqlalchemy as sqla
import decouple, shutil
from IPython.display import display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

engine = sqla.create_engine(decouple.config("MIS_DB"))

In [ ]:
# Old Sell-In Query

query = '''
    -- OUTRIGHT FROM QB
    WITH sales_tbl AS (
        SELECT *, ROW_NUMBER() OVER() AS id
        FROM ( 
            SELECT * FROM sales.or_qb
            UNION ALL
            SELECT * FROM staging.or_qb_mtd
        ) sin
    ),

    join_cte AS (
        SELECT
            s.id, s.`date`, YEAR(s.`date`) AS `year`, MONTHNAME(s.`date`) AS `month`, s.`type`, s.num, s.po_num, ref_an.account_name,
            ref_ad.account_chain, ref_ad.bpc_region, ref_ad.account_channel, ref_ad.channel_class, ref_ad.business_unit, ref_ad.account_type,
            ref_ad.sales_group, ref_l.lead_name, s.memo, s.item, s.item_desc, ref_pc.product_code, ref_pd.product_name, ref_pd.brand,
            ref_pd.packaging, ref_pd.volume, ref_pd.product_class, ref_pd.`usage`, ref_pd.product_type, ref_pd.product_category, s.`account`,
            s.class, s.rep, s.qty, s.um, s.amount, s.sales_tax_code, s.inventory_site, s.ship_to_address_1, s.ship_to_address_2, s.net_amount
        FROM sales_tbl AS s

        -- Add corrected name
        LEFT JOIN ref.account_names AS ref_an
            ON s.`name` = ref_an.`name`

        -- Add account details
        LEFT JOIN ref.account_details AS ref_ad
            ON ref_an.account_name = ref_ad.account_name

        -- Add assigned team leader
        LEFT JOIN ref.lead_names AS ref_l
            ON ref_ad.lead_id = ref_l.lead_id
                AND YEAR(s.`date`) = ref_l.`year`
                AND MONTHNAME(s.`date`) = ref_l.`month`

        -- Add product code
        LEFT JOIN ref.product_codes AS ref_pc
            ON s.item = ref_pc.item

        -- Add product details
        LEFT JOIN ref.product_details AS ref_pd
            ON ref_pc.product_code = ref_pd.product_code1
    ),

    union_cte AS (
        SELECT *, CONCAT(account_name, "-", product_code, "-", brand, "-", um) AS um_key, 1 AS `key_order` FROM join_cte
        UNION ALL
        SELECT *, CONCAT(account_name, "-", brand, "-", um), 2 FROM join_cte
        UNION ALL
        SELECT *, CONCAT(product_code, "-", brand, "-", um), 3 FROM join_cte
        UNION ALL
        SELECT *, CONCAT(brand, "-", um), 4 FROM join_cte
    ),

    partition_cte AS (
        SELECT * 
        FROM (
            SELECT 
                union_cte.*,
                ref_um.um_multiplier,
                ROW_NUMBER() OVER(PARTITION BY union_cte.id ORDER BY union_cte.key_order) AS um_rank
            FROM union_cte
            LEFT JOIN ref.product_ums AS ref_um
                ON union_cte.um_key = ref_um.um_key
            WHERE ref_um.um_key IS NOT NULL
            ) sub
        WHERE um_rank = 1
    )

    SELECT
        `date`, `year`, `month`, "Sell-In" AS `type`, num, po_num, inventory_site, account_name, account_chain, ship_to_address_1, ship_to_address_2, rep, 
        sales_group, lead_name, bpc_region, account_channel, channel_class, business_unit, account_type, item, product_name, product_code, 
        brand, product_class, `usage` AS product_usage, product_type, product_category, um, qty, qty * um_multiplier AS qty_pcs, amount, net_amount
    FROM partition_cte

    UNION ALL

    -- CONSIGNMENT FROM QB
    SELECT
        STR_TO_DATE(CONCAT(cn.`month`, "-1-",YEAR(cn.`date`)), "%M-%d-%Y") AS `date`, YEAR(cn.`date`) AS `year`, cn.`month`, "Sell-Out" AS `type`,
        cn.num, cn.po_num, cn.inventory_site, ref_an.account_name, ref_ad.account_chain, cn.ship_to_address_1, cn.ship_to_address_2, cn.rep, ref_ad.sales_group,
        ref_l.lead_name, ref_ad.bpc_region, ref_ad.account_channel, ref_ad.channel_class, ref_ad.business_unit, ref_ad.account_type, cn.item,
        ref_pd.product_name, ref_pc.product_code, ref_pd.brand, ref_pd.product_class, ref_pd.`usage` AS product_usage, ref_pd.product_type,
        ref_pd.product_category, cn.um, qty, cn.qty AS qty_pcs, cn.amount, cn.net_amount
    FROM (
        SELECT *, ROW_NUMBER() OVER() AS id FROM sales.cn_qb AS sub
    ) cn

    -- Add corrected name
    LEFT JOIN ref.account_names AS ref_an
        ON cn.`name` = ref_an.`name`

    -- Add account details
    LEFT JOIN ref.account_details AS ref_ad
        ON ref_an.account_name = ref_ad.account_name

    -- Add assigned team leader
    LEFT JOIN ref.lead_names AS ref_l
        ON ref_ad.lead_id = ref_l.lead_id
            AND YEAR(cn.`date`) = ref_l.`year`
            AND cn.`month` = ref_l.`month`

    -- Add product code
    LEFT JOIN ref.product_codes AS ref_pc
        ON cn.item = ref_pc.item

    -- Add product details
    LEFT JOIN ref.product_details AS ref_pd
        ON ref_pc.product_code = ref_pd.product_code1

    UNION ALL

    -- CONSIGNMENT FROM PORTAL
    SELECT
        STR_TO_DATE(CONCAT(cn.`month`, "-1-", cn.`year`), "%M-%d-%Y") AS `date`, cn.`year`, cn.`month`, "Sell-Out" AS `type`, 
        NULL AS num, NULL AS po_num, cn.store_name AS inventory_site, cn.account_name, ref_ad.account_chain, cn.store_name AS ship_to_address_1, NULL AS ship_to_address_2, 
        NULL AS rep, ref_ad.sales_group, ref_l.lead_name, ref_ad.bpc_region, ref_ad.account_channel, ref_ad.channel_class, ref_ad.business_unit,
        ref_ad.account_type, NULL AS item, ref_pd.product_name, ref_ic.product_code, ref_pd.brand, ref_pd.product_class, ref_pd.`usage` AS product_usage, 
        ref_pd.product_type, ref_pd.product_category, "PCS" AS um, cn.qty AS `qty`, cn.qty AS qty_pcs, cn.amount, cn.net_amount

    FROM (
        SELECT *, ROW_NUMBER() OVER() AS id FROM sales.cn_prtl AS sub
    ) cn

    -- Add corrected name
    LEFT JOIN ref.account_names AS ref_an
        ON cn.account_name = ref_an.`name`

    -- Add account details
    LEFT JOIN ref.account_details AS ref_ad
        ON ref_an.account_name = ref_ad.account_name

    -- Add assigned team leader
    LEFT JOIN ref.lead_names AS ref_l
        ON ref_ad.lead_id = ref_l.lead_id
            AND cn.`year` = ref_l.`year`
            AND cn.`month` = ref_l.`month`

    -- Add product code
    LEFT JOIN ref.item_codes AS ref_ic
        ON ref_ad.account_chain = ref_ic.account_chain
        AND cn.item_code = ref_ic.item_code

    -- Add product details
    LEFT JOIN ref.product_details AS ref_pd
        ON ref_ic.product_code = ref_pd.product_code1;
'''

In [ ]:
# New Sell-In Query
# ── 1. Pull raw sales tables ───────────────────────────────────────────────────
or_qb     = pd.read_sql("SELECT * FROM sales.or_qb", engine)
or_qb_mtd = pd.read_sql("SELECT * FROM staging.or_qb_mtd", engine)
cn_qb     = pd.read_sql("SELECT * FROM sales.cn_qb", engine)
cn_prtl   = pd.read_sql("SELECT * FROM sales.cn_prtl", engine)

# ── 2. Pull reference tables ───────────────────────────────────────────────────
ref_an = pd.read_sql("SELECT * FROM ref.account_names", engine)
ref_ad = pd.read_sql("SELECT * FROM ref.account_details", engine)
ref_l  = pd.read_sql("SELECT * FROM ref.lead_names", engine)
ref_pc = pd.read_sql("SELECT * FROM ref.product_codes", engine)
ref_pd = pd.read_sql("SELECT * FROM ref.product_details", engine)
ref_um = pd.read_sql("SELECT * FROM ref.product_ums", engine)
ref_ic = pd.read_sql("SELECT * FROM ref.item_codes", engine)
ref_tp = pd.read_sql("SELECT * FROM ref.target_products", engine)

ref_l["year"] = ref_l["year"].astype(int)

# ── sub_tt: reusable NPI target_type lookup ────────────────────────────────────
sub_tt = (
    ref_tp[ref_tp["target_type"] == "NPI"][["year", "account_chain", "product_code", "target_type"]]
    .drop_duplicates()
    .sort_values("target_type")
    .groupby(["year", "account_chain", "product_code"], sort=False)
    .first()
    .reset_index()
)


# ── 3. OUTRIGHT FROM QB ────────────────────────────────────────────────────────
# Mirrors:
#   WITH sales_tbl AS (SELECT *, ROW_NUMBER() OVER() AS id FROM (or_qb UNION ALL or_qb_mtd))

sales_tbl = pd.concat([or_qb, or_qb_mtd], ignore_index=True)
sales_tbl["id"]    = sales_tbl.index
sales_tbl["year"]  = pd.to_datetime(sales_tbl["date"]).dt.year
sales_tbl["month"] = pd.to_datetime(sales_tbl["date"]).dt.strftime("%B")

# LEFT JOIN ref.account_names AS ref_an ON s.name = ref_an.name
# → brings in ref_an.account_name (corrected name)
# Slim ref_an to just the two columns needed so account_name doesn't collide
# if the source table already has an account_name column
_ref_an = ref_an[["name", "account_name"]].copy()
sales_tbl = sales_tbl.drop(columns=["account_name"], errors="ignore")
step1 = sales_tbl.merge(_ref_an, on="name", how="left")

# LEFT JOIN ref.account_details AS ref_ad ON ref_an.account_name = ref_ad.account_name
step2 = step1.merge(ref_ad, on="account_name", how="left")

# LEFT JOIN ref.lead_names AS ref_l
#   ON ref_ad.lead_id = ref_l.lead_id
#  AND YEAR(s.date)       = ref_l.year
#  AND MONTHNAME(s.date)  = ref_l.month
step3 = step2.merge(ref_l, on=["lead_id", "year", "month"], how="left")

# LEFT JOIN ref.product_codes AS ref_pc ON s.item = ref_pc.item
step4 = step3.merge(ref_pc, on="item", how="left")

# LEFT JOIN ref.product_details AS ref_pd ON ref_pc.product_code = ref_pd.product_code1
step5 = step4.merge(ref_pd, left_on="product_code", right_on="product_code1", how="left")

# sub_tt join (target_type override)
join_cte = step5.merge(
    sub_tt, on=["year", "account_chain", "product_code"], how="left", suffixes=("", "_tt")
)
join_cte["target_type"] = join_cte["target_type_tt"].combine_first(join_cte["target_type"])
join_cte = join_cte.drop(columns=["target_type_tt"])

# union_cte: four um_key variants, mirroring the four CONCAT(...) expressions in SQL
# CONCAT(account_name, "-", product_code, "-", brand, "-", um)
u1 = join_cte.copy()
u1["um_key"]    = u1["account_name"].fillna("") + "-" + u1["product_code"].fillna("") + "-" + u1["brand"].fillna("") + "-" + u1["um"].fillna("")
u1["key_order"] = 1

# CONCAT(account_name, "-", brand, "-", um)
u2 = join_cte.copy()
u2["um_key"]    = u2["account_name"].fillna("") + "-" + u2["brand"].fillna("") + "-" + u2["um"].fillna("")
u2["key_order"] = 2

# CONCAT(product_code, "-", brand, "-", um)
u3 = join_cte.copy()
u3["um_key"]    = u3["product_code"].fillna("") + "-" + u3["brand"].fillna("") + "-" + u3["um"].fillna("")
u3["key_order"] = 3

# CONCAT(brand, "-", um)
u4 = join_cte.copy()
u4["um_key"]    = u4["brand"].fillna("") + "-" + u4["um"].fillna("")
u4["key_order"] = 4

union_cte = pd.concat([u1, u2, u3, u4], ignore_index=True)

# partition_cte:
#   LEFT JOIN ref.product_ums ON union_cte.um_key = ref_um.um_key
#   WHERE ref_um.um_key IS NOT NULL   (inner filter — only rows that matched)
#   ROW_NUMBER() OVER(PARTITION BY id ORDER BY key_order) → keep rank = 1
partition_cte = union_cte.merge(ref_um[["um_key", "um_multiplier"]], on="um_key", how="left")
partition_cte = partition_cte[partition_cte["um_multiplier"].notna()]
partition_cte = partition_cte.sort_values(["id", "key_order"])
partition_cte = partition_cte.groupby("id", sort=False).first().reset_index()

# Final SELECT for Sell-In
or_result = partition_cte[[
    "date", "year", "month", "num", "po_num", "inventory_site", "account_name", "account_chain",
    "ship_to_address_1", "ship_to_address_2", "rep", "sales_group", "lead_name", "bpc_region",
    "account_channel", "channel_class", "business_unit", "account_type", "item", "product_name",
    "product_code", "brand", "product_class", "usage", "product_type", "product_category",
    "target_type", "um", "qty", "amount", "net_amount", "um_multiplier"
]].copy()
or_result["type"]    = "Sell-In"
or_result["qty_pcs"] = or_result["qty"] * or_result["um_multiplier"]
or_result = or_result.drop(columns=["um_multiplier"])
or_result = or_result.rename(columns={"usage": "product_usage"})


# ── 4. CONSIGNMENT FROM QB ─────────────────────────────────────────────────────
# Mirrors:
#   SELECT STR_TO_DATE(CONCAT(cn.month, "-1-", YEAR(cn.date)), "%M-%d-%Y") AS date,
#          YEAR(cn.date) AS year, cn.month, "Sell-Out" AS type, ...
#   FROM (SELECT *, ROW_NUMBER() OVER() AS id FROM sales.cn_qb) cn
#   LEFT JOIN ref.account_names  ON cn.name         = ref_an.name
#   LEFT JOIN ref.account_details ON ref_an.account_name = ref_ad.account_name
#   LEFT JOIN ref.lead_names      ON lead_id + year + month
#   LEFT JOIN ref.product_codes   ON cn.item         = ref_pc.item
#   LEFT JOIN ref.product_details ON product_code    = product_code1

cn_qb = cn_qb.copy()
cn_qb["year"]  = pd.to_datetime(cn_qb["date"]).dt.year
cn_qb["date"]  = pd.to_datetime(
    cn_qb["month"] + "-1-" + cn_qb["year"].astype(str), format="%B-%d-%Y"
)

cn_qb = cn_qb.drop(columns=["account_name"], errors="ignore")
cq1 = cn_qb.merge(_ref_an, on="name", how="left")
cq2 = cq1.merge(ref_ad, on="account_name", how="left")
cq3 = cq2.merge(ref_l,  on=["lead_id", "year", "month"], how="left")
cq4 = cq3.merge(ref_pc, on="item", how="left")
cq5 = cq4.merge(ref_pd, left_on="product_code", right_on="product_code1", how="left")
cn_qb_result = cq5.merge(
    sub_tt, on=["year", "account_chain", "product_code"], how="left", suffixes=("", "_tt")
)
cn_qb_result["target_type"] = cn_qb_result["target_type_tt"].combine_first(cn_qb_result["target_type"])
cn_qb_result = cn_qb_result.drop(columns=["target_type_tt"])

# qty_pcs = qty (consignment already in PCS; SQL uses cn.qty for both qty and qty_pcs)
cn_qb_result["type"]    = "Sell-Out"
cn_qb_result["qty_pcs"] = cn_qb_result["qty"]
cn_qb_result = cn_qb_result.rename(columns={"usage": "product_usage"})


# ── 5. CONSIGNMENT FROM PORTAL ─────────────────────────────────────────────────
# Mirrors:
#   SELECT STR_TO_DATE(CONCAT(cn.month, "-1-", cn.year), "%M-%d-%Y") AS date, ...
#          cn.store_name AS inventory_site, cn.account_name, ...
#   FROM (SELECT *, ROW_NUMBER() OVER() AS id FROM sales.cn_prtl) cn
#   LEFT JOIN ref.account_names   ON cn.account_name  = ref_an.name
#   LEFT JOIN ref.account_details ON ref_an.account_name = ref_ad.account_name
#   LEFT JOIN ref.lead_names      ON lead_id + year + month
#   LEFT JOIN ref.item_codes      ON account_chain + item_code
#   LEFT JOIN ref.product_details ON product_code = product_code1

cn_prtl = cn_prtl.copy()
cn_prtl["year"] = cn_prtl["year"].astype(int)
cn_prtl["date"] = pd.to_datetime(
    cn_prtl["month"] + "-1-" + cn_prtl["year"].astype(str), format="%B-%d-%Y"
)

# SQL: cn.account_name = ref_an.name  (portal's account_name is the raw/uncorrected name)
# Then downstream uses ref_an.account_name (the corrected name) for all subsequent joins.
# Rename to avoid collision so ref_an.account_name is the only account_name after the merge.
cn_prtl = cn_prtl.rename(columns={"account_name": "cn_account_name"})

pp1 = cn_prtl.merge(_ref_an, left_on="cn_account_name", right_on="name", how="left")
pp2 = pp1.merge(ref_ad, on="account_name", how="left")
pp3 = pp2.merge(ref_l,  on=["lead_id", "year", "month"], how="left")
pp4 = pp3.merge(ref_ic, on=["account_chain", "item_code"], how="left")
pp5 = pp4.merge(ref_pd, left_on="product_code", right_on="product_code1", how="left")
cn_prtl_result = pp5.merge(
    sub_tt, on=["year", "account_chain", "product_code"], how="left", suffixes=("", "_tt")
)
cn_prtl_result["target_type"] = cn_prtl_result["target_type_tt"].combine_first(cn_prtl_result["target_type"])
cn_prtl_result = cn_prtl_result.drop(columns=["target_type_tt"])

# Literal columns from SQL SELECT (NULL AS num, cn.store_name AS inventory_site, etc.)
cn_prtl_result["type"]              = "Sell-Out"
cn_prtl_result["qty_pcs"]           = cn_prtl_result["qty"]
cn_prtl_result["um"]                = "PCS"
cn_prtl_result["num"]               = None
cn_prtl_result["po_num"]            = None
cn_prtl_result["item"]              = None
cn_prtl_result["rep"]               = None
cn_prtl_result["inventory_site"]    = cn_prtl_result["store_name"]
cn_prtl_result["ship_to_address_1"] = cn_prtl_result["store_name"]
cn_prtl_result["ship_to_address_2"] = None
cn_prtl_result = cn_prtl_result.rename(columns={"usage": "product_usage"})


# ── 6. UNION ALL three results ─────────────────────────────────────────────────
final_cols = [
    "date", "year", "month", "type", "num", "po_num", "inventory_site", "account_name",
    "account_chain", "ship_to_address_1", "ship_to_address_2", "rep", "sales_group", "lead_name",
    "bpc_region", "account_channel", "channel_class", "business_unit", "account_type", "item",
    "product_name", "product_code", "brand", "product_class", "product_usage", "product_type",
    "product_category", "target_type", "um", "qty", "qty_pcs", "amount", "net_amount"
]

sin_df = pd.concat(
    [or_result[final_cols], cn_qb_result[final_cols], cn_prtl_result[final_cols]],
    ignore_index=True
)

sin_df["date"] = pd.to_datetime(sin_df["date"]).dt.date

sin_df.head()

In [ ]:
# ── 3. OUTRIGHT FROM QB ────────────────────────────────────────────────────────
# Mirrors:
#   WITH sales_tbl AS (SELECT *, ROW_NUMBER() OVER() AS id FROM (or_qb UNION ALL or_qb_mtd))

sales_tbl = pd.concat([or_qb, or_qb_mtd], ignore_index=True)
sales_tbl["id"]    = sales_tbl.index
sales_tbl["year"]  = pd.to_datetime(sales_tbl["date"]).dt.year
sales_tbl["month"] = pd.to_datetime(sales_tbl["date"]).dt.strftime("%B")

# LEFT JOIN ref.account_names AS ref_an ON s.name = ref_an.name
# → brings in ref_an.account_name (corrected name)
# Slim ref_an to just the two columns needed so account_name doesn't collide
# if the source table already has an account_name column
_ref_an = ref_an[["name", "account_name"]].copy()
sales_tbl = sales_tbl.drop(columns=["account_name"], errors="ignore")
step1 = sales_tbl.merge(_ref_an, on="name", how="left")

# LEFT JOIN ref.account_details AS ref_ad ON ref_an.account_name = ref_ad.account_name
step2 = step1.merge(ref_ad, on="account_name", how="left")

# LEFT JOIN ref.lead_names AS ref_l
#   ON ref_ad.lead_id = ref_l.lead_id
#  AND YEAR(s.date)       = ref_l.year
#  AND MONTHNAME(s.date)  = ref_l.month
step3 = step2.merge(ref_l, on=["lead_id", "year", "month"], how="left")

# LEFT JOIN ref.product_codes AS ref_pc ON s.item = ref_pc.item
step4 = step3.merge(ref_pc, on="item", how="left")

# LEFT JOIN ref.product_details AS ref_pd ON ref_pc.product_code = ref_pd.product_code1
step5 = step4.merge(ref_pd, left_on="product_code", right_on="product_code1", how="left")

In [ ]:
# Write Sell-In Report

mis_fldr_path = decouple.config("MIS_FILE_PATH")
sin_df.to_excel(rf"{mis_fldr_path}Report Templates\Generated Sell-In YTD.xlsx", index=False)
# sin_df.to_excel(r"C:\eli_dump\Backup\Database Files\YTD Reports\Generated Sell-In YTD.xlsx", index=False)
shutil.copy(rf"{mis_fldr_path}Report Templates\Generated Sell-In YTD.xlsx", r"C:\eli_dump\Backup\Database Files\YTD Reports\Generated Sell-In YTD.xlsx")